# RLM vs RAG vs naive — интерактивный прогон

Запуск стенда из Jupyter с живым наблюдением за генерацией RLM.

**Перед запуском:**
1. Поднять модель (LM Studio или vLLM) и при необходимости указать конфиг.
2. Для RAG с локальным эмбеддером — скачать модель: `python -m scripts.download_embedder`.
3. Установить зависимости: `pip install -r requirements.txt` (нужен `ipywidgets`).

Конфиг: оставь поле `config` пустым (возьмётся `config.yaml` или `$RLM_CONFIG`) или укажи путь, напр. `config.vllm.yaml`.

In [ ]:
# Запускать из корня репозитория. Если ноутбук лежит в notebooks/ — поднимемся на уровень выше.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Рабочая папка:", os.getcwd())

In [ ]:
from eval.notebook_ui import launch

# В форме два режима:
#  1) «Синтетический датасет» — прежний прогон (кнопка «Запустить»).
#  2) «Реальные документы» — 5 строк «папка базы знаний + свой вопрос».
#     Пустые строки пропускаются. Поддержка: docx, xlsx, csv, pptx, pdf, txt, md.
#     Вся папка склеивается в один контекст; эталон не нужен — ответы методов
#     ложатся рядом для ручного сравнения. Результат → results_real.{json,xlsx}.
# quick=True — быстрый синтетический прогон (на режим документов не влияет).
ui = launch(config_path="", dataset="simple", quick=True)

## Просмотр результатов

- **Синтетика:** `results.json` / `results.xlsx` (или `results_complex.*`) — ячейка ниже считает accuracy.
- **Реальные документы:** `results_real.json` / `results_real.xlsx` — эталона нет, поэтому смотри ответы методов рядом (последняя ячейка) или открой `results_real.xlsx` (листы «Сравнение» и «Ресурсы»).

In [ ]:
import json
import pandas as pd

path = "results.json"  # или results_complex.json
recs = json.load(open(path, encoding="utf-8"))["records"]
df = pd.DataFrame(recs)
df["tokens"] = df["usage"].apply(lambda u: u.get("total_tokens", 0))
summary = df.groupby(["method", "type"]).agg(
    n=("correct", "size"),
    accuracy=("correct", "mean"),
    avg_sec=("elapsed", "mean"),
    avg_tokens=("tokens", "mean"),
).round(2)
summary

In [ ]:
# Реальные документы: ответы методов рядом (эталона нет).
import json
import pandas as pd

recs = json.load(open("results_real.json", encoding="utf-8"))["records"]
df = pd.DataFrame(recs)
df["tokens"] = df["usage"].apply(lambda u: u.get("total_tokens", 0))
# Сводная: строки — (папка, вопрос), колонки — ответы методов.
pivot = df.pivot_table(index=["source", "question"], columns="method",
                       values="answer", aggfunc="first")
pivot